# SC-VAE — آموزش و ارزیابی روی Google Colab

کلون مخزن ← نصب ← دادهٔ کوچک ← آموزش (نمایش همگرایی) ← نمودار همگرایی ← ارزیابی (PSNR/SSIM/LPIPS + بازسازی).

> ⚙️ قبل از اجرا: **Runtime → Change runtime type → GPU** (مثلاً T4).


## ۱) دریافت کد و نصب وابستگی‌ها

In [ ]:
!git clone https://github.com/e-shams-d/SC-VAE.git

In [ ]:
import os
os.chdir('/content/SC-VAE')
!pip install -q omegaconf easydict tensorboardX

## ۲) آماده‌سازی دادهٔ کوچک

یک مجموعهٔ کوچکِ **ثابت** از تصاویر واقعی ۲۵۶×۲۵۶ دانلود می‌کنیم و فایل‌های لیستِ مخزن را بازنویسی می‌کنیم.

> برای **آموزش واقعی**، این سلول را با تصاویر واقعی FFHQ جایگزین کن (در `dataset/FFHQ/resized/`).


In [ ]:
import os, requests

root = 'dataset/FFHQ/resized'
os.makedirs(root, exist_ok=True)

N = 64
ok = []
for i in range(N):
    name = f'{i:05d}.jpg'
    try:
        r = requests.get(f'https://picsum.photos/seed/{i}/256/256',
                         headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        if r.status_code == 200 and len(r.content) > 1000:
            open(f'{root}/{name}', 'wb').write(r.content)
            ok.append(name)
    except Exception as e:
        print('skip', name, e)

# rewrite the repo list files so they reference ONLY the images we actually have
for fn in ['ffhqtrain.txt', 'ffhqvalidation.txt']:
    open(f'img_datasets/assets/{fn}', 'w').write('\n'.join(ok) + '\n')

print(f'downloaded {len(ok)}/{N} images; train/val list files rewritten')

## ۳) آموزش

اجرای سبکِ نمایشِ همگرایی (روی T4 حدود ۶–۸ دقیقه):

| پارامتر | مقدار | توضیح |
|---------|-------|-------|
| `experiment.epochs` | 30 | برای دیدن روند کاهش loss |
| `experiment.batch_size` | 4 | امن برای حافظهٔ T4 |
| `optimizer.init_lr` | 2.0e-4 | یادگیری محسوس (gradient clipping مانع انفجار است) |
| `optimizer.warmup.min_lr` | 1.0e-5 | کاهش تدریجی LR |

👀 خط‌های `[epoch N] validation reconstruction loss:` باید **کاهش** پیدا کنند.
🆘 اگر `NaN` دیدی، `optimizer.init_lr=2.0e-4` را به `1.0e-4` کم کن.


In [ ]:
!python main-stage1.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --device cuda --num-workers 2 \
  experiment.epochs=30 experiment.batch_size=4 \
  optimizer.init_lr=2.0e-4 optimizer.warmup.min_lr=1.0e-5

## ۴) checkpointها + TensorBoard

In [ ]:
!ls -lh results/models/FFHQ/*/
%load_ext tensorboard
%tensorboard --logdir results/logs

## ۵) نمودار همگرایی

منحنیِ loss را مستقیم از لاگ‌های TensorBoard می‌خوانیم و با matplotlib رسم می‌کنیم
(بدون هیچ تغییری در کد آموزش/تست).


In [ ]:
import glob, os
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# آخرین پوشهٔ لاگ
log_dir = sorted(glob.glob('results/logs/FFHQ/*'), key=os.path.getmtime)[-1]
ea = EventAccumulator(log_dir, size_guidance={'scalars': 0})
ea.Reload()

def scal(tag):
    ev = ea.Scalars(tag)
    return [e.step for e in ev], [e.value for e in ev]

tags = ea.Tags()['scalars']
plt.figure(figsize=(8, 5))
if 'loss/train/reconstruction' in tags:
    st, vt = scal('loss/train/reconstruction')
    plt.plot(st, vt, alpha=0.25, label='train recon (per-batch)')
if 'loss/test/reconstruction' in tags:
    s, v = scal('loss/test/reconstruction')
    plt.plot(s, v, marker='o', linewidth=2, label='validation recon (per-epoch)')
plt.xlabel('training step'); plt.ylabel('reconstruction loss (MSE)')
plt.title('SC-VAE convergence'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('results/convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved results/convergence.png')

## ۶) ارزیابی (PSNR / SSIM / LPIPS + بازسازی)

`evaluate.py` همان مدل SC-VAE و همان pipeline دادهٔ آموزش/تست را می‌سازد، وزنِ `best.pt` را بار می‌کند،
و معیارهای بازسازیِ مقاله را محاسبه می‌کند (روش از `reconstruction.py`ِ اصلی گرفته شده است).


In [ ]:
!pip install -q piq lpips

import glob, os
# آخرین checkpoint از اجرای آموزش
ckpt = sorted(glob.glob('results/models/FFHQ/*/best.pt'), key=os.path.getmtime)[-1]
print('using checkpoint:', ckpt)

!python evaluate.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --load-path "{ckpt}" --device cuda --batch-size 8

In [ ]:
# نمایش مقایسهٔ اصل (ردیف بالا) و بازسازی (ردیف پایین)
from IPython.display import Image
Image('results/eval/reconstruction_compare.png')